Mục tiêu chính của phần demo này là để bạn có một hình dung sơ bộ về cách mình sẽ song song hóa một tác vụ để có thể tăng tốc (đối với các bạn đã học về CUDA C/C++ thì mình cũng hy vọng bạn sẽ thấy sự dễ dàng khi code bằng CUDA Python). Nếu bạn không hiểu rõ ý nghĩa của một số dòng code thì cũng không sao cả; mình sẽ giải thích rõ hơn ở các buổi tới.

Tác vụ mà mình sẽ làm ở phần demo này là nhân hai ma trận A và B (nhân theo kiểu đại số tuyến tính).

In [ ]:
!pip uninstall -y numba llvmlite numpy

In [ ]:
!pip install -q numba==0.61.0 llvmlite==0.44.0

In [ ]:
import numpy as np
import math
from numba import cuda

In [ ]:
A = np.random.randint(-100, 100, (1000, 1500))
B = np.random.randint(-100, 100, (1500, 2000))

In [ ]:
A

array([[-93, -41, -28, ..., -58,   9, -46],
       [-34, -21,  89, ...,   7,  15, -15],
       [ 96, -60,  64, ...,  57, -69, -64],
       ...,
       [  3, -31, -96, ..., -38, -41, -58],
       [-66, -87,  45, ...,  99,  95, -18],
       [-78,  41, -85, ..., -50,  72, -85]])

In [ ]:
B

array([[-79,  32,  45, ...,  93, -63,  23],
       [-16, -96,  60, ...,  98, -41, -62],
       [ 53,  83, -42, ...,  89,  21, -20],
       ...,
       [-73, -34,  11, ...,  12, -34, -74],
       [ 23,  76,  96, ..., -85,  85, -22],
       [-98,  95,  88, ..., -83,  46, -50]])

In [ ]:
A.shape

(1000, 1500)

In [ ]:
B.shape

(1500, 2000)

In [ ]:
# Cờ C_CONTIGUOUS bằng True nghĩa là mảng được lưu bằng cách lấy các dòng và nối lại với nhau
A.flags

  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False
  UPDATEIFCOPY : False

**Cài đặt tuần tự trên host (CPU) bằng Python**

In [ ]:
def mul_mat_python(A, B, C):
    for r in range(C.shape[0]):
        for c in range(C.shape[1]):
            temp = 0
            for i in range(A.shape[1]):
                temp += A[r, i] * B [i, c]
            C[r, c] = temp

In [ ]:
C_python = np.empty((A.shape[0], B.shape[1]), dtype=int)

In [ ]:
%%time
mul_mat_python(A, B, C_python)
# Nếu bạn không đủ kiên nhẫn để chờ thì có thể ấn Ctrl-M + I để Interrupt

KeyboardInterrupt: ignored

**Cài đặt tuần tự trên host (CPU) bằng Numpy**

In [ ]:
%%time
C_numpy = A @ B

CPU times: user 5.96 s, sys: 13.9 ms, total: 5.97 s
Wall time: 5.94 s


In [ ]:
# np.mean(np.abs(C_numpy - C_python))

**Cài đặt song song trên device (GPU) bằng Cuda**

Do việc tính mỗi phần tử trong ma trận kết quả là độc lập với nhau nên ta có thể để mỗi thread bên GPU phụ trách tính một phần tử trong ma trận kết quả và các thread này sẽ cùng chạy song song với nhau.

In [ ]:
@cuda.jit
def mul_mat_kernel(A, B, C):
    r = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    c = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if r < C.shape[0] and c < C.shape[1]:
        temp = 0
        for i in range(A.shape[1]):
            temp += A[r, i] * B[i, c]
        C[r, c] = temp

In [ ]:
C_cuda = np.empty((A.shape[0], B.shape[1]), dtype=int)
block_size = (16, 16)
grid_size = (math.ceil(C_cuda.shape[1] / block_size[0]),
             math.ceil(C_cuda.shape[0] / block_size[1]))

In [ ]:
%%time
mul_mat_kernel[grid_size, block_size](A, B, C_cuda)
# Lần chạy đầu tiên sẽ lâu do phải biên dịch ra mã máy rồi mới chạy
# Từ lần chạy thứ 2 trở đi sẽ nhanh hơn do không phải biên dịch
# mà chỉ việc chạy mã máy đã được biên dịch và đã được cache lại trước đó

CPU times: user 80.7 ms, sys: 869 µs, total: 81.6 ms
Wall time: 84.2 ms


Mình chạy thì thấy việc cài đặt song song trên GPU bằng Cuda (lần chạy thứ 2, không phải biên dịch ra mã máy) giúp tăng tốc 49 lần so với cài đặt tuần tự trên CPU bằng Numpy!

In [ ]:
np.mean(np.abs(C_cuda - C_numpy))

0.0

**Cài đặt song song trên device (GPU) bằng Cuda**

In [ ]:
%%time

import cupy as cp

# Transfer to GPU
A_gpu = cp.asarray(A)
B_gpu = cp.asarray(B)

# Multiply
C_gpu = A_gpu @ B_gpu

# Bring result back to CPU
C = cp.asnumpy(C_gpu)

In [ ]:
print("Result shape:", C.shape)
print("Sample value C[0,0]:", C[0, 0])

# Verify against NumPy
C_ref = A @ B
print("Correct:", np.allclose(C, C_ref))